## Myanmar ACLED Exploration Notebook

Exploratory analysis of ACLED conflict event data. **All analysis is scoped to Myanmar only.** The filter is applied in §3 (Data Cleaning) and propagates through the rest of the notebook.

## 1. Setup

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from sklearn.linear_model import LogisticRegression

# File paths and output directories
DATA_PATH = Path('../../data/raw/acled_myanmar_2026-04-22.csv')
ACTOR_INV_DIR = Path('actor_inventory')
ACTOR_INV_DIR.mkdir(exist_ok=True)

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 120)

## 2. Data Loading

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print(f'Loaded {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns')
print(f'\nColumn dtypes:')
print(df_raw.dtypes.to_string())
print(f'\nCountries in file: {df_raw["country"].value_counts().to_dict()}')

## 3. Data Cleaning

In [ ]:
df = df_raw.copy()

# Parse event_date to datetime
df['event_date'] = pd.to_datetime(df['event_date'])

# === SCOPE: filter to Myanmar only — all downstream cells use this df ===
df = df[df['country'] == 'Myanmar'].copy()
print(f'Myanmar events: {len(df):,}')

# Missing-value report
missing = df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print('\nMissing values per column:')
print(missing.to_string())

print(f'\nDate range: {df["event_date"].min().date()} → {df["event_date"].max().date()}')

## 4. Initial Exploration

All counts below are Myanmar-only.

In [ ]:
print('=== Event types ===')
print(df['event_type'].value_counts().to_string())

print('\n=== Sub-event types ===')
print(df['sub_event_type'].value_counts().to_string())

print('\n=== Admin1 regions ===')
print(df['admin1'].value_counts().to_string())

print(f'\nTotal events:     {len(df):,}')
print(f'Total fatalities: {df["fatalities"].sum():,}')
print(f'Date range:       {df["event_date"].min().date()} → {df["event_date"].max().date()}')
print(f'Years:            {sorted(df["year"].unique())}')

## 5. Event Type Classification

Map each `sub_event_type` to its ACLED parent category and flag territorial-control signals.

In [ ]:
print(df.sub_event_type.value_counts())

# Classification of sub_event_type into ACLED's official parent event types:
# Battles, Violence against civilians, Explosions/Remote violence,
# Strategic developments, Protests, Riots
def classify_sub_event_type(sub_event_type):
    battles = [
        'Armed clash',
        'Government regains territory',
        'Non-state actor overtakes territory'
    ]

    violence_against_civilians = [
        'Attack',
        'Abduction/forced disappearance',
        'Sexual violence'
    ]

    remote_violence = [
        'Air/drone strike',
        'Shelling/artillery/missile attack',
        'Remote explosive/landmine/IED',
        'Grenade',
        'Suicide bomb',
        'Chemical weapon'
    ]

    strategic_developments = [
        'Agreement',
        'Arrests',
        'Change to group/activity',
        'Disrupted weapons use',
        'Headquarters or base established',
        'Non-violent transfer of territory',
        'Looting/property destruction',
        'Other'
    ]

    protests = [
        'Peaceful protest',
        'Protest with intervention',
        'Excessive force against protesters'
    ]

    riots = [
        'Violent demonstration',
        'Mob violence'
    ]

    if sub_event_type in battles:
        return 'Battles'
    elif sub_event_type in violence_against_civilians:
        return 'Violence against civilians'
    elif sub_event_type in remote_violence:
        return 'Explosions/Remote violence'
    elif sub_event_type in strategic_developments:
        return 'Strategic developments'
    elif sub_event_type in protests:
        return 'Protests'
    elif sub_event_type in riots:
        return 'Riots'
    else:
        return 'Unknown'

df['event_category'] = df['sub_event_type'].apply(classify_sub_event_type)
print(df['event_category'].value_counts())

# Sanity check — make sure nothing fell into 'Unknown'
print('\nUnclassified:', df[df['event_category'] == 'Unknown']['sub_event_type'].unique())

# Territorial control signal flag
control_signals = [
    'Non-state actor overtakes territory',
    'Government regains territory',
    'Non-violent transfer of territory',
    'Headquarters or base established'
]
df['is_control_signal'] = df['sub_event_type'].isin(control_signals)
print(f'\nTerritorial control signal events: {df["is_control_signal"].sum()}')

## 6. Temporal Exploration

In [ ]:
# ---------- Plot 1: one line per event_category over time (plotly) ----------
monthly = (
    df.groupby([pd.Grouper(key='event_date', freq='ME'), 'event_category'])
      .size()
      .reset_index(name='count')
)

fig1 = px.line(
    monthly,
    x='event_date',
    y='count',
    color='event_category',
    title='Myanmar — Events over time by category',
    labels={'event_date': 'Date', 'count': 'Number of events (monthly)', 'event_category': 'Category'}
)
fig1.update_layout(
    hovermode='x unified',
    width=1200,
    height=600,
    legend=dict(orientation='v', yanchor='middle', y=0.5, xanchor='left', x=1.02)
)
fig1.show()

In [ ]:
# ---------- Plot 2: static matplotlib version for publication ----------
monthly_mat = (
    df.groupby([pd.Grouper(key='event_date', freq='ME'), 'event_category'])
      .size()
      .reset_index(name='count')
)

fig, ax = plt.subplots(figsize=(14, 5))
cat_colors = {
    'Battles': '#1f77b4',
    'Explosions/Remote violence': '#d62728',
    'Strategic developments': '#2ca02c',
    'Protests': '#ff7f0e',
    'Violence against civilians': '#9467bd',
    'Riots': '#8c564b',
}
for cat, grp in monthly_mat.groupby('event_category'):
    ax.plot(grp['event_date'], grp['count'],
            label=cat, color=cat_colors.get(cat), linewidth=1.4)

ax.set_title('Myanmar — Events over time by category (monthly)', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel('Events per month')
ax.legend(loc='upper left', fontsize=9, framealpha=0.7)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.grid(axis='y', alpha=0.3)
ax.axvline(pd.Timestamp('2021-02-01'), color='black', linestyle='--', linewidth=1, alpha=0.6, label='Feb 2021 coup')
ax.annotate('Feb 2021\ncoup', xy=(pd.Timestamp('2021-02-01'), ax.get_ylim()[1] * 0.85),
            fontsize=8, color='black', ha='left')
plt.tight_layout()
plt.show()

In [ ]:
# ---------- Plot: territorial control signal events only ----------
monthly_control = (
    df[df['is_control_signal']]
      .groupby([pd.Grouper(key='event_date', freq='ME'), 'sub_event_type'])
      .size()
      .reset_index(name='count')
)

fig2 = px.line(
    monthly_control,
    x='event_date',
    y='count',
    color='sub_event_type',
    title='Myanmar — Territorial control signal events over time',
    labels={'event_date': 'Date', 'count': 'Number of events (monthly)', 'sub_event_type': 'Sub-event type'},
    markers=True
)
fig2.update_layout(
    hovermode='x unified',
    width=1200,
    height=550,
    legend=dict(orientation='v', yanchor='middle', y=0.5, xanchor='left', x=1.02)
)
fig2.show()

In [ ]:
# ---------- Fatalities as background, control signals as foreground ----------
monthly_fatalities = (
    df.groupby(pd.Grouper(key='event_date', freq='ME'))['fatalities']
      .sum()
)

monthly_ctrl = (
    df[df['is_control_signal']]
      .groupby([pd.Grouper(key='event_date', freq='ME'), 'sub_event_type'])
      .size()
      .unstack(fill_value=0)
)

fig = make_subplots(specs=[[{'secondary_y': True}]])

fig.add_trace(
    go.Scatter(
        x=monthly_fatalities.index,
        y=monthly_fatalities.values,
        name='Fatalities (total)',
        fill='tozeroy',
        line=dict(color='rgba(150, 150, 150, 0.4)', width=0),
        fillcolor='rgba(150, 150, 150, 0.25)',
        hovertemplate='%{x|%b %Y}<br>Fatalities: %{y:,}<extra></extra>'
    ),
    secondary_y=True
)

control_colors = {
    'Non-state actor overtakes territory': '#D85A30',
    'Government regains territory': '#185FA5',
    'Non-violent transfer of territory': '#1D9E75',
    'Headquarters or base established': '#7F77DD'
}

for sub_event in monthly_ctrl.columns:
    fig.add_trace(
        go.Scatter(
            x=monthly_ctrl.index,
            y=monthly_ctrl[sub_event],
            name=sub_event,
            mode='lines+markers',
            line=dict(color=control_colors.get(sub_event, '#444'), width=2),
            marker=dict(size=5),
            hovertemplate='%{x|%b %Y}<br>' + sub_event + ': %{y}<extra></extra>'
        ),
        secondary_y=False
    )

fig.update_layout(
    title='Myanmar — Territorial control events vs total fatalities over time',
    hovermode='x unified',
    width=1300, height=600,
    legend=dict(orientation='h', y=-0.15),
    plot_bgcolor='white'
)
fig.update_yaxes(title_text='Control-signal events (count)', secondary_y=False,
                 showgrid=True, gridcolor='rgba(0,0,0,0.05)')
fig.update_yaxes(title_text='Fatalities (monthly total)', secondary_y=True, showgrid=False)
fig.update_xaxes(title_text='Date', showgrid=False)
fig.show()

In [ ]:
def build_monthly(df, country=None):
    sub = df if country is None else df[df['country'] == country]
    monthly = (
        sub.groupby([pd.Grouper(key='event_date', freq='ME'), 'event_category'])
           .size()
           .unstack(fill_value=0)
    )
    control = (
        sub[sub['is_control_signal']]
           .groupby(pd.Grouper(key='event_date', freq='ME'))
           .size()
    )
    monthly['Control_signals'] = control.reindex(monthly.index, fill_value=0)
    return monthly

# Since df is already filtered to Myanmar, pass without the country filter
country = 'Myanmar'
monthly = build_monthly(df)
print(monthly.tail())

In [ ]:
fig = make_subplots(specs=[[{'secondary_y': True}]])

fig.add_trace(
    go.Scatter(x=monthly.index, y=monthly['Battles'],
               name='Battles', line=dict(color='steelblue', width=1.5)),
    secondary_y=False
)
fig.add_trace(
    go.Bar(x=monthly.index, y=monthly['Control_signals'],
           name='Control changes', marker_color='crimson', opacity=0.7),
    secondary_y=True
)

fig.update_layout(
    title=f'Myanmar — Battles vs territorial control events over time',
    hovermode='x unified',
    width=1200, height=500,
    legend=dict(orientation='h', y=1.1)
)
fig.update_yaxes(title_text='Battles (monthly count)', secondary_y=False)
fig.update_yaxes(title_text='Control events (monthly count)', secondary_y=True)
fig.show()

In [ ]:
battles = monthly['Battles']
control = monthly['Control_signals']

lags = range(-6, 7)  # battles 6 months before to 6 months after control changes
correlations = []
for lag in lags:
    if lag >= 0:
        corr = battles.shift(lag).corr(control)
    else:
        corr = battles.corr(control.shift(-lag))
    correlations.append(corr)

lag_df = pd.DataFrame({'lag_months': list(lags), 'correlation': correlations})

fig = go.Figure(go.Bar(
    x=lag_df['lag_months'], y=lag_df['correlation'],
    marker_color=['steelblue' if l >= 0 else 'lightgray' for l in lag_df['lag_months']]
))
fig.add_hline(y=0, line_dash='dash', line_color='black')
fig.update_layout(
    title=f'Myanmar — Cross-correlation of battles and control events',
    xaxis_title='Lag (months) — positive = battles lead control changes',
    yaxis_title='Correlation',
    width=900, height=400
)
fig.show()
print(lag_df.to_string(index=False))

In [ ]:
# Quick lagged correlation example
monthly_flat = df.set_index('event_date').groupby(
    [pd.Grouper(freq='ME'), 'event_category']
).size().unstack(fill_value=0)
battles_ts = monthly_flat['Battles']
control_ts = df[df['is_control_signal']].set_index('event_date').groupby(
    pd.Grouper(freq='ME')
).size()
control_ts = control_ts.reindex(battles_ts.index, fill_value=0)

for lag in range(0, 7):
    print(f'Lag {lag} months: corr = {battles_ts.corr(control_ts.shift(-lag)):.3f}')

# Grid-based spatial approach
df['lat_bin'] = (df['latitude'] // 0.5) * 0.5
df['lon_bin'] = (df['longitude'] // 0.5) * 0.5

cell_stats = df.groupby(['lat_bin', 'lon_bin']).agg(
    battles=('event_category', lambda x: (x == 'Battles').sum()),
    control_events=('is_control_signal', 'sum')
).reset_index()

print(cell_stats[['battles', 'control_events']].corr())

X = cell_stats[['battles']].values
y = (cell_stats['control_events'] > 0).astype(int).values
model = LogisticRegression().fit(X, y)
print(f'Logistic coefficient: {model.coef_[0][0]:.4f}')

## 7. Actor and Interaction Exploration

Full inventory of actors, interaction codes, and actor-type codes. Summary CSVs are saved to `actor_inventory/`.

In [ ]:
# Quick reconnaissance — actor-related columns
print('Columns available:', [c for c in df.columns if 'actor' in c.lower() or 'inter' in c.lower()])
print('\nSample rows:')
print(df[['sub_event_type', 'actor1', 'actor2', 'interaction']].head(10))
print(f'\nActor1 unique count: {df["actor1"].nunique()}')
print(f'Actor2 unique count: {df["actor2"].nunique()} (+ {df["actor2"].isna().sum():,} NaN = events with no actor2)')
print('Interaction column dtype:', df['interaction'].dtype)
print('Interaction unique values:', sorted(df['interaction'].dropna().unique()))

In [ ]:
# ---------- 1. All unique actor1 values ----------
print('='*70)
print(f'ACTOR1 — {df["actor1"].nunique()} unique values')
print('='*70)
actor1_counts = df['actor1'].value_counts(dropna=False)
print(actor1_counts.to_string())

# ---------- 2. All unique actor2 values ----------
print('\n' + '='*70)
print(f'ACTOR2 — {df["actor2"].nunique()} unique values')
print('='*70)
actor2_counts = df['actor2'].value_counts(dropna=False)
print(actor2_counts.to_string())

# ---------- 3. All unique interaction labels ----------
print('\n' + '='*70)
print(f'INTERACTION — {df["interaction"].nunique()} unique values')
print('='*70)
interaction_counts = df['interaction'].value_counts(dropna=False)
print(interaction_counts.to_string())

# ---------- 4. inter1 and inter2 individually ----------
if 'inter1' in df.columns and 'inter2' in df.columns:
    print('\n' + '='*70)
    print('INTER1 (actor1 type) and INTER2 (actor2 type)')
    print('='*70)
    print('\ninter1:')
    print(df['inter1'].value_counts(dropna=False).sort_index().to_string())
    print('\ninter2:')
    print(df['inter2'].value_counts(dropna=False).sort_index().to_string())

In [ ]:
# Save actor inventory CSVs to actor_inventory/ subfolder (Myanmar-only)

# actor1 inventory
actor1_inv = (
    df.groupby('actor1')
      .agg(
          event_count=('event_id_cnty', 'count'),
          inter1=('inter1', 'first'),
          top_admin1=('admin1', lambda x: x.value_counts().index[0]),
          top_sub_event=('sub_event_type', lambda x: x.value_counts().index[0]),
          total_fatalities=('fatalities', 'sum'),
      )
      .reset_index()
      .sort_values('event_count', ascending=False)
)
actor1_inv.to_csv(ACTOR_INV_DIR / 'actor1_inventory.csv', index=False)
print(f'Saved actor1 inventory: {len(actor1_inv)} unique actors')
print(actor1_inv.head(20).to_string(index=False))

# actor2 inventory
actor2_inv = (
    df[df['actor2'].notna()]
      .groupby('actor2')
      .agg(
          event_count=('event_id_cnty', 'count'),
          inter2=('inter2', 'first'),
      )
      .reset_index()
      .sort_values('event_count', ascending=False)
)
actor2_inv.to_csv(ACTOR_INV_DIR / 'actor2_inventory.csv', index=False)
print(f'\nSaved actor2 inventory: {len(actor2_inv)} unique actors')

# interaction dyad inventory
dyad_inv = (
    df.groupby(['actor1', 'actor2', 'interaction'])
      .agg(event_count=('event_id_cnty', 'count'), total_fatalities=('fatalities', 'sum'))
      .reset_index()
      .sort_values('event_count', ascending=False)
)
dyad_inv.to_csv(ACTOR_INV_DIR / 'dyad_inventory.csv', index=False)
print(f'\nSaved dyad inventory: {len(dyad_inv)} unique dyads')

# interaction-type summary
interaction_inv = (
    df.groupby(['inter1', 'inter2', 'interaction'])
      .agg(event_count=('event_id_cnty', 'count'))
      .reset_index()
      .sort_values('event_count', ascending=False)
)
interaction_inv.to_csv(ACTOR_INV_DIR / 'interaction_inventory.csv', index=False)
print(f'Saved interaction inventory: {len(interaction_inv)} unique interaction types')

## 8. Per-Sub-Event Actor Analysis

Focus sub-events: **Armed clash**, **Air/drone strike**, **Arrests**, **Abduction/forced disappearance**.

In [ ]:
target_subs = ['Armed clash', 'Air/drone strike', 'Arrests', 'Abduction/forced disappearance']
focus = df[df['sub_event_type'].isin(target_subs)].copy()

print(f'Total events in focus subset: {len(focus):,}')
print('\nBreakdown by sub-event:')
print(focus['sub_event_type'].value_counts())

# Top 15 actor1 per sub-event type
for sub in target_subs:
    print(f"\n{'='*60}\nTop actor1 in '{sub}':")
    print(focus[focus['sub_event_type'] == sub]['actor1'].value_counts().head(15))

top_n = 10
top_actors = (
    focus.groupby(['sub_event_type', 'actor1'])
         .size()
         .reset_index(name='count')
         .sort_values(['sub_event_type', 'count'], ascending=[True, False])
         .groupby('sub_event_type')
         .head(top_n)
)

fig = px.bar(
    top_actors,
    x='count', y='actor1',
    color='sub_event_type',
    facet_col='sub_event_type', facet_col_wrap=2,
    orientation='h',
    height=800, width=1300,
    title=f'Myanmar — Top {top_n} actor1 by sub-event type'
)
fig.update_yaxes(matches=None, autorange='reversed')
fig.update_xaxes(matches=None)
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# Use inter1 and inter2 columns directly (text labels in this ACLED data version)
focus['inter1_label'] = focus['inter1']
focus['inter2_label'] = focus['inter2'].fillna('(none)')

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=target_subs,
    horizontal_spacing=0.15, vertical_spacing=0.18
)

for i, sub in enumerate(target_subs):
    sub_df = focus[focus['sub_event_type'] == sub]
    pivot = (
        sub_df.groupby(['inter1_label', 'inter2_label'])
              .size()
              .unstack(fill_value=0)
    )
    row, col = (i // 2) + 1, (i % 2) + 1
    fig.add_trace(
        go.Heatmap(
            z=pivot.values,
            x=pivot.columns.tolist(), y=pivot.index.tolist(),
            colorscale='Blues',
            showscale=False,
            text=pivot.values, texttemplate='%{text}',
            hovertemplate='Actor1: %{y}<br>Actor2: %{x}<br>Count: %{z}<extra></extra>'
        ),
        row=row, col=col
    )

fig.update_layout(
    height=800, width=1300,
    title_text='Myanmar — Actor type interaction heatmaps (who does what to whom)'
)
fig.update_xaxes(tickangle=45)
fig.show()

In [ ]:
# Top dyads per sub-event type
top_n_dyad = 10
focus['dyad'] = focus['actor1'] + ' → ' + focus['actor2'].fillna('(no actor2)')

top_dyads = (
    focus.groupby(['sub_event_type', 'dyad'])
         .size()
         .reset_index(name='count')
         .sort_values(['sub_event_type', 'count'], ascending=[True, False])
         .groupby('sub_event_type')
         .head(top_n_dyad)
)

fig = px.bar(
    top_dyads,
    x='count', y='dyad',
    color='sub_event_type',
    facet_col='sub_event_type', facet_col_wrap=2,
    orientation='h',
    height=900, width=1400,
    title=f'Myanmar — Top {top_n_dyad} actor dyads by sub-event type'
)
fig.update_yaxes(matches=None, autorange='reversed')
fig.update_xaxes(matches=None)
fig.for_each_annotation(lambda a: a.update(text=a.text.split('=')[-1]))
fig.update_layout(showlegend=False)
fig.show()

In [ ]:
# Top-5 actor1 time series per focus sub-event type
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=target_subs,
    horizontal_spacing=0.10, vertical_spacing=0.18,
    shared_xaxes=False
)

for i, sub in enumerate(target_subs):
    sub_df = focus[focus['sub_event_type'] == sub].copy()
    top5 = sub_df['actor1'].value_counts().head(5).index.tolist()

    monthly_sub = (
        sub_df[sub_df['actor1'].isin(top5)]
        .groupby([pd.Grouper(key='event_date', freq='ME'), 'actor1'])
        .size()
        .reset_index(name='count')
    )

    row, col = (i // 2) + 1, (i % 2) + 1
    for actor in top5:
        actor_ts = monthly_sub[monthly_sub['actor1'] == actor]
        # Truncate long actor names for legend readability
        short_name = actor.split(':')[0].split('(')[0].strip()[:35]
        fig.add_trace(
            go.Scatter(
                x=actor_ts['event_date'],
                y=actor_ts['count'],
                name=short_name,
                mode='lines',
                line=dict(width=1.5),
                showlegend=(i == 0),
                hovertemplate=actor + '<br>%{x|%b %Y}: %{y}<extra></extra>'
            ),
            row=row, col=col
        )

    fig.update_xaxes(title_text='Date', row=row, col=col)
    fig.update_yaxes(title_text='Events/month', row=row, col=col)

fig.update_layout(
    height=800, width=1300,
    title_text='Myanmar — Top-5 actor1 monthly activity by sub-event type',
    hovermode='x unified'
)
fig.show()

## 9. Proposed Actor Grouping

### 9.1 Actor Landscape Analysis (Myanmar)

#### Unique actor counts and distribution shape

The Myanmar ACLED data contains **2,050 unique actor1 values** and **808 unique actor2 values** (plus ~32,500 events with no actor2, i.e., ~31% of all events involve only one actor). Both distributions are strongly long-tailed:

- The **top 3 actor1 entries** account for ~64% of all events: Military Forces of Myanmar (2021-) with 42,827, Protesters (Myanmar) with 18,163, and Unidentified Armed Group (Myanmar) with 6,990.
- The **top 10 actor1 entries** account for ~83% of events; the remaining 2,040 actors share the bottom 17%.
- This is a classic "fat head / long tail" distribution: a handful of actors dominate the record, but hundreds of local or ephemeral actors scatter across the tail — many are local PDF units ("People's Defense Force — [township]"), each counted separately.

#### Interaction code dominance and dyad structure

The text-based `interaction` column (e.g., "State forces-Civilians") encodes the same information as the inter1/inter2 pair. The dominant dyads are:

| Interaction | Count | % | Interpretation |
|---|---|---|---|
| State forces-Civilians | 26,527 | 25% | Tatmadaw arresting, attacking, abducting civilians |
| State forces-Political militia | 21,518 | 20% | Tatmadaw vs PDF (ACLED codes PDF as political militia) |
| Protesters only | 17,237 | 16% | Civilian protests (no opposing actor) |
| State forces-Rebel group | 14,378 | 14% | Tatmadaw vs ethnic armed organizations (EAOs) |
| State forces only | 9,642 | 9% | Unilateral state actions |
| Political militia-Civilians | 5,785 | 5% | PDF/militias vs civilians |

These six types cover **~90% of all Myanmar events**. The conflict is therefore not diffuse — it is dominated by a star-shaped structure where the Tatmadaw (State forces) sits at the center of most dyads.

#### Per-sub-event actor structure

**Armed clash (27,115 events):** The clearest bilateral structure. Two main opponent pairs:
- Tatmadaw vs PDF/resistance (State forces-Political militia: 13,107)
- Tatmadaw vs EAOs (State forces-Rebel group: 12,094)
Named EAO actors appear prominently as both actor1 and actor2: KIA, AA, KNU/KNLA, TNLA, MNDAA. Actor1 is roughly split 44% State forces vs 37% Political militia vs 19% Rebel group.

**Air/drone strike (9,721 events):** Highly concentrated. The Tatmadaw alone accounts for 7,127 events (73%) as actor1. Most strikes are against unspecified targets; "State forces only" (3,868) means strikes with no identified target. The remainder is Tatmadaw vs civilians (2,778) or vs PDF (2,188). A small but analytically interesting group of PDF drone units (Myaing, Yesagyo, MDDS, etc.) have emerged since 2022 as actor1 in counter-strikes.

**Arrests (6,335 events):** Almost entirely State forces-Civilians (6,150 = 97%). Dominated by a single actor: Military Forces of Myanmar (2021-) with 5,967 events. Police Forces of Myanmar (2021-) account for 261. This sub-event type is essentially a single bilateral pattern: junta security forces detaining civilians.

**Abduction/forced disappearance (3,125 events):** More fragmented than arrests. Tatmadaw dominates (1,979 = 63%) but EAOs and militias account for a meaningful share: AA/ULA (154), TNLA (79), KIA (64), SSPP/SSA-N (63), MNDAA (51), Pyu Saw Htee (43). Interaction is almost always vs Civilians. This is the sub-event type where EAO behavior toward civilians is most analytically visible.

#### Named actors that matter most — and their data quirks

| Group | Key actor strings in ACLED | Notes |
|---|---|---|
| **Tatmadaw** | "Military Forces of Myanmar (2021-)", "(2016-2021)", "(2011-2016)", "(1988-2011)" | Same institution, 4 era-based names — must be consolidated |
| **Police** | "Police Forces of Myanmar (2021-)", "(2016-2021)", "(2011-2016)" | Era-split; functionally Tatmadaw-allied |
| **PDF/Anti-Coup resistance** | "PDF: People's Defense Force" + hundreds of "People's Defense Force — [township]" variants + "Unidentified Anti-Coup Armed Group" | ~200+ distinct entries all representing post-2021 NUG-aligned resistance |
| **KIA** | "KIO/KIA: Kachin Independence Organization/Kachin Independence Army" | Consistently formatted |
| **AA** | "ULA/AA: United League of Arakan/Arakan Army" | Consistently formatted |
| **KNU/KNLA** | "KNU/KNLA: Karen National Union/Karen National Liberation Army" | Consistently formatted |
| **TNLA** | "PSLF/TNLA: Palaung State Liberation Front/Ta'ang National Liberation Army" | Consistently formatted |
| **MNDAA** | "MNTJP/MNDAA: Myanmar National Truth and Justice Party/Myanmar National Democratic Alliance Army" | Long; active in Shan-North/Kokang |
| **Pyu Saw Htee** | "Pyu Saw Htee" | Pro-military arson/intimidation militia; coded as Identity militia |
| **Civilians** | "Civilians (Myanmar)", "Protesters (Myanmar)", "Rioters (Myanmar)" | Non-combatant categories |

**Key naming quirks:**
- The era-splits on Tatmadaw/Police are the most important normalization need: ~48,500 events across 4 military entries and ~900 across 3 police entries all refer to the same chain of command.
- PDF fragmentation is severe: over 200 local PDF entries (by township, district, and region). ACLED follows the "People's Defense Force — [location]" naming convention for locally formed units. Together they represent tens of thousands of events but appear as tiny tail entries individually.
- "Unidentified Anti-Coup Armed Group" (1,569 actor1 events) is a catch-all for resistance actors the ACLED coders could not identify — likely predominantly PDF or allied militias.
- "Unidentified Armed Group (Myanmar)" (6,990 events) is a broader catch-all that spans both sides of the conflict.

#### Admin1 regional actor mix

| Region | Events | Dominant actor1 | Distinct character |
|---|---|---|---|
| **Sagaing** | 26,182 (25%) | Tatmadaw + many local PDFs | Epicenter of post-coup PDF resistance; many local PDF units |
| **Magway** | 9,707 (9%) | Tatmadaw + local PDFs | Second PDF heartland |
| **Mandalay** | 9,519 (9%) | Tatmadaw + local PDFs + Protesters | Mixed: PDF resistance + urban protests |
| **Shan-North** | 8,357 (8%) | TNLA, MNDAA, Tatmadaw | Three Brotherhood Alliance theater; EAO-dominated |
| **Rakhine** | 7,282 (7%) | AA, Tatmadaw | AA-vs-Tatmadaw bilateral, very little PDF |
| **Kachin** | 7,082 (7%) | KIA, Tatmadaw | KIA-vs-Tatmadaw bilateral, predates 2021 coup |
| **Yangon** | 6,374 (6%) | Tatmadaw + Protesters | Predominantly protests and arrests, minimal armed clashes |

Sagaing+Magway+Mandalay (43% of events) are the PDF resistance heartland. Rakhine and Kachin have qualitatively different actor mixes — bilateral EAO conflicts with far less PDF presence. Shan-North is its own theater dominated by the Three Brotherhood Alliance (Operation 1027). Yangon is protest-dominated. Any model that pools across admin1 without controlling for actor mix will conflate very different conflict dynamics.

### 9.2 Proposed Grouping Scheme

#### Options reviewed

| Option | Description | Pros | Cons |
|---|---|---|---|
| (a) inter1 type only | 8 buckets by actor type code | Clean, generalizable | Loses all named-actor detail; conflates PDF and Pyu Saw Htee under same bucket |
| (b) Top-N named + "Other [type]" | Keep top N actors, lump rest | Preserves key actors | Arbitrary N; still hides Tatmadaw era-split |
| (c) Interaction dyad | State-rebel, state-civilian, etc. | Standard in quant conflict lit | Dyad is relationship not actor; can't assign a single group to an event where actor1 can be on either side |
| (d) Hybrid top-N + dyad residual | Named actors for big players, dyad for rest | Flexible | Complex; still requires Myanmar-aware top-N selection |
| **(e) Myanmar coalition grouping** | Domain-aware coalitions reflecting conflict structure | Best fits Myanmar's post-2021 landscape; analytically interpretable | Requires manual curation; not portable to other countries |

#### Recommendation: Option (e) — Myanmar-tailored coalition grouping

**Proposed groups:**

| Group label | Actor strings to match |
|---|---|
| `Tatmadaw` | All "Military Forces of Myanmar (*)" |
| `State Security` | All "Police Forces of Myanmar (*)", "Government of Myanmar (*)" |
| `Pro-Junta Militias` | Pyu Saw Htee, People's Militia Force, Border Guard Force (junta-affiliated), other pro-junta identity militias |
| `PDF & Anti-Coup Resistance` | All "People's Defense Force — [location]", "PDF: People's Defense Force", "Unidentified Anti-Coup Armed Group", other post-2021 resistance |
| `KIA` | "KIO/KIA: Kachin Independence Organization/Kachin Independence Army" |
| `AA` | "ULA/AA: United League of Arakan/Arakan Army" |
| `KNU/KNLA` | "KNU/KNLA: Karen National Union/Karen National Liberation Army" |
| `TNLA` | "PSLF/TNLA: Palaung State Liberation Front/Ta'ang National Liberation Army" |
| `MNDAA` | "MNTJP/MNDAA: Myanmar National Truth and Justice Party/Myanmar National Democratic Alliance Army" |
| `Other EAOs` | RCSS/SSA-S, SSPP/SSA-N, KNDF, CNF/CNA, KNPP/KA, PNLO/PNLA, DKBA, NA-B, etc. |
| `Protesters` | "Protesters (Myanmar)", "Rioters (Myanmar)" |
| `Civilians` | "Civilians (Myanmar)" |
| `Unidentified` | "Unidentified Armed Group (Myanmar)", other unidentified |
| `Other` | All remaining actors |

**Justification (why option e, why these buckets):**

The generic inter1 actor-type codes (option a) are insufficient for Myanmar because the "Political militia" bucket alone hides the single most important analytical distinction in the data: Pyu Saw Htee (pro-junta arson militia) and the PDF (anti-junta resistance) are coded identically. Running a territorial-control model on "political militia" events would average across two opposing sides, producing meaningless estimates. Option (e) resolves this by treating the post-coup alignment structure as the primary axis of grouping.

Among the EAOs, keeping KIA, AA, KNU/KNLA, TNLA, and MNDAA as named groups rather than collapsing them into "Other EAOs" matters because: (1) they are regionally concentrated and thus map to distinct geographic theaters; (2) Operation 1027 (October 2023) — the major offensive that dramatically shifted territorial control in Shan-North and Rakhine — was led by AA+TNLA+MNDAA as the Three Brotherhood Alliance, and KIA coordinated in Kachin. A generic "EAO" bucket would make this event invisible in the actor dimension. The five named EAOs together account for ~7,000 actor1 events (7% of total); the remaining EAOs (~20 groups) together account for ~2,000 events and are safely lumped.

**Trade-offs:** This grouping enables direct alignment-level analysis (junta coalition vs resistance coalition vs neutral civilians) and region-specific EAO attribution — the most important features for a territorial-control model. What it loses is portability: the function will be Myanmar-specific and cannot be reused for Somalia, Nigeria, or Ecuador without rewriting. It also requires ongoing maintenance if ACLED's naming conventions change or new actors emerge. Given that this project is scoped to Myanmar for the thesis, these are acceptable costs.

## 10. Actor Grouping — Implementation

Approved scheme from §9. Adds three nested columns: `side` -> `coalition` -> `actor_group`, plus `is_mapped` (True = matched a named rule, False = fell to inter1 fallback or Unmapped).

In [ ]:
import re

# ══════════════════════════════════════════════════════════════════════════════
# ACTOR GROUPING CONFIGURATION
# Each rule: patterns (list of regexes, case-insensitive, OR-matched against actor1)
#             side, coalition, actor_group
# Rules applied in ORDER — first match wins; most specific rules must come FIRST.
# Time-varying overrides are applied post-hoc via TIME_RULES (see below).
# ══════════════════════════════════════════════════════════════════════════════

ACTOR_RULES = [

    # ── ANTI-JUNTA ▸ National Unity Government (PDF, NUG) ────────────────────
    {
        "patterns": [
            r"people.s\s+def",            # People's Defense Force (all local variants)
            r"people\s+def(?:ense|ence)\s+force",
            r"\bpdf\b",                   # PDF: People's Defense Force (generic entry)
            r"unidentified anti.coup",
            r"people.s security force",
            r"natogyi people",
            r"kani people.s def",
        ],
        "side": "Anti-junta",
        "coalition": "National Unity Government (PDF, NUG)",
        "actor_group": "People's Defence Force",
    },

    # ── ANTI-JUNTA ▸ Northern Alliance ───────────────────────────────────────
    {
        "patterns": [r"kio/kia", r"kachin independence"],
        "side": "Anti-junta",
        "coalition": "Northern Alliance",
        "actor_group": "Kachin Independence Army (KIA)",
    },
    # Northern Alliance collective label -> attributed to KIA as leading member
    {
        "patterns": [r"^na-b:.*northern alliance"],
        "side": "Anti-junta",
        "coalition": "Northern Alliance",
        "actor_group": "Kachin Independence Army (KIA)",
    },

    # ── ANTI-JUNTA ▸ Three Brotherhood Alliance ───────────────────────────────
    {
        "patterns": [r"ula/aa", r"united league.*arakan", r"arakan army"],
        "side": "Anti-junta",
        "coalition": "Three Brotherhood Alliance",
        "actor_group": "Arakan Army (AA)",
    },
    {
        "patterns": [r"pslf/tnla", r"ta.?ang national lib", r"palaung state lib"],
        "side": "Anti-junta",
        "coalition": "Three Brotherhood Alliance",
        "actor_group": "TNLA",
        # time override: after 2026-01-01 -> Other / Unmapped / Other combatant / TNLA
    },
    {
        "patterns": [r"mntjp/mndaa", r"myanmar national.*alliance army",
                     r"myanmar national truth and justice"],
        "side": "Anti-junta",
        "coalition": "Three Brotherhood Alliance",
        "actor_group": "MNDAA",
        # time override: after 2026-01-01 -> Other / Unmapped / Other combatant / MNDAA
    },
    # Three Brotherhood Alliance collective label (events coded to the coalition)
    {
        "patterns": [r"^brotherhood alliance$"],
        "side": "Anti-junta",
        "coalition": "Three Brotherhood Alliance",
        "actor_group": "Three Brotherhood Alliance (collective)",
    },

    # ── ANTI-JUNTA ▸ 4K Coalition ────────────────────────────────────────────
    {
        "patterns": [
            r"knu/knla", r"karen national union", r"karen national lib(?:eration)? army",
            r"\bkndo\b", r"karen national def.*org",
            r"kaw thoo lei", r"kayin force",
        ],
        "side": "Anti-junta",
        "coalition": "4K Coalition",
        "actor_group": "Karen National Liberation Army (KNLA)",
    },
    {
        "patterns": [r"knpp/ka", r"karenni national progressive", r"\bkarenni army\b"],
        "side": "Anti-junta",
        "coalition": "4K Coalition",
        "actor_group": "Karenni Army",
    },
    {
        "patterns": [r"\bkndf\b", r"karenni nationalities def"],
        "side": "Anti-junta",
        "coalition": "4K Coalition",
        "actor_group": "KNDF",
    },
    {
        "patterns": [r"\bknplf\b", r"karenni nationalities people.s lib"],
        "side": "Anti-junta",
        "coalition": "4K Coalition",
        "actor_group": "KNPLF",
        # time override: before 2023-01-01 -> Other / Unmapped / Unmapped / KNPLF
    },

    # ── ANTI-JUNTA ▸ Chinland Council ────────────────────────────────────────
    {
        "patterns": [
            r"cnf/cna", r"chin national front",
            r"chinland def(?:ense|ence) force", r"cdf-kkg",
            r"acdf.*asho chin",
        ],
        "side": "Anti-junta",
        "coalition": "Chinland Council",
        "actor_group": "Chinland Council",
    },

    # ── ANTI-JUNTA ▸ Chin Brotherhood Alliance ───────────────────────────────
    {
        "patterns": [
            r"chin brotherhood", r"cno/cndf",
            r"chin national org.*def", r"chin national def.*force",
            r"zomi federal union", r"pdf.*zoland", r"zoland def",
        ],
        "side": "Anti-junta",
        "coalition": "Chin Brotherhood Alliance",
        "actor_group": "Chin Brotherhood Alliance",
    },

    # ── ANTI-JUNTA ▸ Spring Revolution Alliance ───────────────────────────────
    {
        "patterns": [r"pnlo/pnla", r"pa-oh national lib"],
        "side": "Anti-junta",
        "coalition": "Spring Revolution Alliance",
        "actor_group": "PNLA",
        # time override: before 2024-01-01 -> Other / Unmapped / Unmapped / PNLA
    },

    # ── ANTI-JUNTA ▸ Other allied EAOs ───────────────────────────────────────
    {
        "patterns": [
            r"nmsp/mnla", r"nmsp-ad", r"mnla-ad", r"mon national lib",
            r"new mon state party",
            r"\bmsrf\b", r"\bmsru\b", r"mon state rev",
            r"mon state mount", r"ramonnya mon",
        ],
        "side": "Anti-junta",
        "coalition": "Other allied EAOs",
        "actor_group": "Mon National Liberation Army (MNLA)",
    },
    {
        "patterns": [r"dkba.*benev", r"democratic karen benev"],
        "side": "Anti-junta",
        "coalition": "Other allied EAOs",
        "actor_group": "Other allied EAOs",
    },

    # ── ANTI-JUNTA ▸ Other anti-junta organisations ───────────────────────────
    {
        "patterns": [r"\babsdf\b", r"all burma students"],
        "side": "Anti-junta",
        "coalition": "Other anti-junta organisations",
        "actor_group": "Other anti-junta",
    },
    {
        "patterns": [
            r"^pla:.*people.s lib", r"people.s liberation army",
            r"anti.fascist.*(?:armed )?force", r"anti.fascist int",
            r"\bpadf\b.*anti.fascist",
        ],
        "side": "Anti-junta",
        "coalition": "Other anti-junta organisations",
        "actor_group": "Other anti-junta",
    },

    # ── ANTI-JUNTA ▸ Local resistance forces (PDF-allied / locally organised) ──
    # ~80 locally named anti-junta militias; all ACLED-coded Political militia.
    # Broad patterns cover common naming conventions; specific names cover unique groups.
    {
        "patterns": [
            # broad naming conventions used by anti-junta local forces
            r"guerr?illa",                                   # guerrilla / guerilla variants
            r"revolution(?:ary)?\s+(?:forces?|army|front|alliance)",
            r"anti.dictatorship",
            r"burma\s+(?:national\s+revolutionary|liberation\s+democratic)",
            r"pyauk\s+kyar",                                # Burmese: "guerrilla"
            r"sit\s+kyaung",                                # Burmese: "column"
            # specific named groups not caught by broad patterns
            r"ye\s+bi\s+lu|ye\s+ogre",                   # Ye Ogre Group (Mon State)
            r"myingyan\s+black\s+tiger|\bmbt\b.*myingyan|myingyan.*\bmbt\b",
            r"\bcdsom\b",
            r"kanbalu.*(?:underground|\bug\b)",
            r"27\s+revolution\s+forces",
            r"\bpafd\b|army\s+to\s+fight\s+dictatorship",
            r"yaw\s+defense\s+force",
            r"black\s+eagle\s+defense",
            r"\bmdds\b|myingyan\s+district\s+drone",
            r"\bugp.okpo\b|underground.*guerr?illa.*okpo",
            r"\bmrda\b|royal\s+dragon\s+army",
            r"\bprdf\b|palaw\s+regional\s+defense",
            r"chindwin\s+attack\s+force",
            r"kyaikhto\s+revolution|\bkrf\b",
            r"generation\s+z\s+power",
            r"young\s+force.ug",
            r"civic\s+defense\s+militia.*siyin",
            r"brave\s+heart\s+army|\bbha\b.*brave",
            r"\bmnrf\b|moe\s+nyo\s+revolution",
            r"phoenix\s+df\b|phoenix\s+defense\s+force.*nattalin",
            r"\badpra\b",
            r"yangon\s+army",
            r"dark\s+shadow",
            r"brave\s+warriors\s+for\s+myanmar|\bbwm\b",
            r"chindwin\s+brothers",
            r"\bblpa\b|black\s+leopard\s+army",
            r"dawei\s+defense\s+team|\bddt\b.*dawei",
            r"people.s\s+revolution\s+army",
            r"danger\s+force\s+ldf",
            r"\bpkdf\b|people.s\s+knight\s+defense",
            r"nat\s+soe\s+myay|demon\s+underground\s+revolutionary",
            r"bo\s+lin\s+yone\s+tatphwe|bo\s+eagle\s+force",
            r"bagan\s+ogre\s+force",
            r"freedom\s+revolution\s+force|\bfrf\b|farmers\s+revolution\s+force",
            r"operation\s+flame",
            r"\bkddf\b|kyaukse\s+district\s+defense",
            r"red\s+bandana\s+column|pu\s+war\s+ni\s+sit",
            r"\bnrdf\b|natogyi\s+regional\s+defense",
            r"\busba\b|united\s+states\s+of\s+burma\s+army",
            r"freeland\s+attack\s+force|\bfla\b.*freeland",
            r"people.s\s+revolution\s+alliance.*magway|pra\s+magway",
            r"\bsstf\b|salingyi\s+special\s+task",
            r"phoenix\s+sgg",
            r"myaung\s+revolutionary\s+army|\bmra\b.*myaung",
            r"\bcgf\b|chauk\s+guerr?illa",
            r"dawei.*joint\s+forces|dawei\s+kha\s+yaing",
            r"myaing\s+villages\s+revolution|\bmvrf\b",
            r"northern\s+thandaung\s+defense|\bntdf\b",
            r"\beagle\s+force\b",
            r"golden\s+eagle\s+force",
            r"people.s\s+servant\s+revolution|\bpsr\b.*wetlet",
            r"shar\s+htoo\s+waw\s+drone",
            r"khin\s+u\s+support|\bkso\b.*khin",
            r"du\s+ya\s+ka\s+(?:sit|column)",
            r"mya\s+nan\s+dar.*(?:mandalay|sit|operation)",
            r"daw\s+na\s+column",
            r"yaw\s+revolution\s+army.*tilin|yra.*tilin",
            r"king\s+cobra.khin\s+u",
            r"thayarwady\s+galon",
            r"taw\s+gyi\s+mway\s+hauk|king\s+cobra\s+force",
            r"\bacplf\b|anti.coup\s+people.s\s+liberation",
            r"gyobingauk\s+(?:hero|guerr?illa)",
            r"\bbldf\b|burma\s+liberation\s+democratic",
            r"brother\s+defense\s+force.*yinmarbin|\bbdf\b.*yinmarbin",
            r"\bmgw\b|magway\s+guerr?illa\s+warfare",
            r"tha\s+pyay\s+nyo\s+(?:guerr?illa|pyauk)",
            r"\bystf\b|ye\s+special\s+task",
            r"aung\s+si\s+taw.*mandalay|\bast.mdy\b",
        ],
        "side": "Anti-junta",
        "coalition": "National Unity Government (PDF, NUG)",
        "actor_group": "People's Defence Force",
    },


    # ── PRO-JUNTA ▸ Tatmadaw and allies ──────────────────────────────────────
    {
        "patterns": [
            r"military forces of myanmar",
            r"police forces of myanmar",
            r"government of myanmar",    # covers SAC + all era variants
        ],
        "side": "Pro-junta",
        "coalition": "Tatmadaw and allies",
        "actor_group": "Tatmadaw",
    },
    {
        "patterns": [r"pyu saw", r"pyusawhti"],
        "side": "Pro-junta",
        "coalition": "Tatmadaw and allies",
        "actor_group": "Pyusawhti militias",
    },
    {
        "patterns": [r"thway thauk aphwe", r"blood comrades.*pro.mil"],
        "side": "Pro-junta",
        "coalition": "Tatmadaw and allies",
        "actor_group": "Thway Thout",
    },

    # ── PRO-JUNTA ▸ Aligned ethnic armed organisations ────────────────────────
    {
        "patterns": [r"^kna:.*karen national army$"],
        "side": "Pro-junta",
        "coalition": "Aligned ethnic armed organisations",
        "actor_group": "KNA",
    },
    {
        "patterns": [r"alp/ala", r"arakan liberation"],
        "side": "Pro-junta",
        "coalition": "Aligned ethnic armed organisations",
        "actor_group": "ALA",
    },
    {
        "patterns": [r"pno/pna", r"pa-oh national org", r"pa.oh national army"],
        "side": "Pro-junta",
        "coalition": "Aligned ethnic armed organisations",
        "actor_group": "PNA",
    },
    {
        "patterns": [r"shanni nationalities", r"\bsna\b.*shanni"],
        "side": "Pro-junta",
        "coalition": "Aligned ethnic armed organisations",
        "actor_group": "Shanni Nationalities Army",
    },
    {
        "patterns": [r"\bzra\b", r"zomi revolutionary army"],
        "side": "Pro-junta",
        "coalition": "Aligned ethnic armed organisations",
        "actor_group": "ZRA",
    },
    {
        "patterns": [r"\barsa\b", r"arakan rohingya salvation"],
        "side": "Pro-junta",
        "coalition": "Aligned ethnic armed organisations",
        "actor_group": "ARSA",
    },
    {
        "patterns": [
            r"kno/kna.*kuki", r"kuki national",
            r"sspp/ssa-n", r"shan state progress",
            r"dkba.*buddh", r"democratic karen buddh",
            r"rohingya muslim militia",
            r"\brso\b.*rohingya", r"rakhine ethnic militia",
            r"thway thauk",      # remaining Thway Thauk variants
        ],
        "side": "Pro-junta",
        "coalition": "Aligned ethnic armed organisations",
        "actor_group": "Other pro-junta EAOs",
    },

    # ── NEUTRAL / SELF-ADMINISTERED ───────────────────────────────────────────
    {
        "patterns": [r"uwsp/uwsa", r"united wa state"],
        "side": "Neutral / Self-administered",
        "coalition": "Self-administered zones",
        "actor_group": "UWSA",
    },
    {
        "patterns": [r"psc/ndaa-ess", r"ndaa.*eastern shan"],
        "side": "Neutral / Self-administered",
        "coalition": "Self-administered zones",
        "actor_group": "NDAA",
    },
    {
        "patterns": [r"rcss/ssa-s", r"restoration council.*shan", r"shan state army.south"],
        "side": "Neutral / Self-administered",
        "coalition": "Self-administered zones",
        "actor_group": "RCSS / Shan State Army South",
    },

    # ── OTHER / UNMAPPED ──────────────────────────────────────────────────────
    {
        "patterns": [r"civilians?\s*\(myanmar\)", r"^civilians?\s*$"],
        "side": "Other / Unmapped",
        "coalition": "Unmapped",
        "actor_group": "Civilians",
    },
    {
        "patterns": [r"protesters?\s*\(myanmar\)", r"^protesters?\s*$"],
        "side": "Other / Unmapped",
        "coalition": "Unmapped",
        "actor_group": "Protesters",
    },
    {
        "patterns": [r"rioters?\s*\(myanmar\)", r"^rioters?\s*$"],
        "side": "Other / Unmapped",
        "coalition": "Unmapped",
        "actor_group": "Protesters",
    },
]

# ── TIME-VARYING OVERRIDES ─────────────────────────────────────────────────────
# Applied after base mapping. "direction": "after" means override fires for
# events >= cutoff; "before" means < cutoff.
TIME_RULES = [
    {   # TNLA left the Three Brotherhood Alliance / rejoined junta ceasefire ~Jan 2026
        "actor_group": "TNLA",   "direction": "after",
        "cutoff": pd.Timestamp("2026-01-01"),
        "override": {"side": "Other / Unmapped", "coalition": "Other combatant",
                     "actor_group": "TNLA"},
    },
    {   # MNDAA same realignment
        "actor_group": "MNDAA",  "direction": "after",
        "cutoff": pd.Timestamp("2026-01-01"),
        "override": {"side": "Other / Unmapped", "coalition": "Other combatant",
                     "actor_group": "MNDAA"},
    },
    {   # KNPLF only joined 4K Coalition from Jan 2023
        "actor_group": "KNPLF",  "direction": "before",
        "cutoff": pd.Timestamp("2023-01-01"),
        "override": {"side": "Other / Unmapped", "coalition": "Unmapped",
                     "actor_group": "KNPLF"},
    },
    {   # PNLA only joined Spring Revolution Alliance from Jan 2024
        "actor_group": "PNLA",   "direction": "before",
        "cutoff": pd.Timestamp("2024-01-01"),
        "override": {"side": "Other / Unmapped", "coalition": "Unmapped",
                     "actor_group": "PNLA"},
    },
]

# ── INTER1-TYPE FALLBACK ───────────────────────────────────────────────────────
# For actor1 strings that match NO named rule, use inter1 to assign a generic bucket.
INTER1_FALLBACK = {
    "State forces":           ("Pro-junta",             "Tatmadaw and allies",             "Tatmadaw"),
    "Rebel group":            ("Other / Unmapped",      "Other combatant",                 "Unmapped"),
    "Political militia":      ("Other / Unmapped",      "Other combatant",                 "Unmapped"),
    "Identity militia":       ("Other / Unmapped",      "Other combatant",                 "Unmapped"),
    "Rioters":                ("Other / Unmapped",      "Unmapped",                        "Protesters"),
    "Protesters":             ("Other / Unmapped",      "Unmapped",                        "Protesters"),
    "Civilians":              ("Other / Unmapped",      "Unmapped",                        "Civilians"),
    "External/Other forces":  ("Other / Unmapped",      "Unmapped",                        "Unmapped"),
}

# Pre-compile for performance
_compiled_rules = [
    {**r, "_pats": [re.compile(p, re.IGNORECASE) for p in r["patterns"]]}
    for r in ACTOR_RULES
]

# Warn about spec'd groups absent from data
_SPEC_GROUPS = [
    "People's Defence Force", "Kachin Independence Army (KIA)", "Arakan Army (AA)",
    "TNLA", "MNDAA", "Three Brotherhood Alliance (collective)",
    "Karen National Liberation Army (KNLA)", "Karenni Army", "KNDF", "KNPLF",
    "Chinland Council", "Chin Brotherhood Alliance", "PNLA",
    "Mon National Liberation Army (MNLA)", "Other allied EAOs", "Other anti-junta",
    "Tatmadaw", "Pyusawhti militias", "Thway Thout",
    "KNA", "ALA", "PNA", "Shanni Nationalities Army", "ZRA", "ARSA",
    "Other pro-junta EAOs",
    "UWSA", "NDAA", "RCSS / Shan State Army South",
    "Civilians", "Protesters", "Unmapped",
]
print(f"Rules loaded: {len(ACTOR_RULES)} named rules, {len(TIME_RULES)} time overrides")


In [ ]:

def _match_actor(actor_str: str):
    # Return (side, coalition, actor_group, is_mapped) for one actor string.
    s = str(actor_str)
    for rule in _compiled_rules:
        if any(p.search(s) for p in rule["_pats"]):
            return rule["side"], rule["coalition"], rule["actor_group"], True
    return None, None, None, False


def group_actors(df: pd.DataFrame) -> pd.DataFrame:
    # Add side, coalition, actor_group, is_mapped columns
    df = df.copy()

    # Step 1 — build lookup over unique actor1 strings (fast path)
    unique_actors = df["actor1"].dropna().unique()
    lookup = {a: _match_actor(a) for a in unique_actors}

    results = df["actor1"].map(lookup)   # Series of 4-tuples or NaN
    df["side"]        = results.map(lambda x: x[0] if isinstance(x, tuple) else None)
    df["coalition"]   = results.map(lambda x: x[1] if isinstance(x, tuple) else None)
    df["actor_group"] = results.map(lambda x: x[2] if isinstance(x, tuple) else None)
    df["is_mapped"]   = results.map(lambda x: x[3] if isinstance(x, tuple) else False)

    # Step 2 — inter1-type fallback for rows still null
    fallback_mask = df["side"].isna()
    for inter1_val, (fb_side, fb_coal, fb_ag) in INTER1_FALLBACK.items():
        m = fallback_mask & (df["inter1"] == inter1_val)
        df.loc[m, "side"]        = fb_side
        df.loc[m, "coalition"]   = fb_coal
        df.loc[m, "actor_group"] = fb_ag
        # is_mapped stays False (set in initial map, not overwritten here)

    # Step 3 — catch any remaining nulls (NaN actor1, unknown inter1)
    still_null = df["side"].isna()
    df.loc[still_null, ["side", "coalition", "actor_group"]] = [
        "Other / Unmapped", "Unmapped", "Unmapped"
    ]
    df.loc[still_null, "is_mapped"] = False

    # Step 4 — time-varying overrides
    for rule in TIME_RULES:
        ag_mask   = df["actor_group"] == rule["actor_group"]
        date_mask = (df["event_date"] >= rule["cutoff"]) if rule["direction"] == "after" \
                    else (df["event_date"] <  rule["cutoff"])
        ov = rule["override"]
        df.loc[ag_mask & date_mask, ["side", "coalition", "actor_group"]] = \
            [ov["side"], ov["coalition"], ov["actor_group"]]

    return df


# Apply
df = group_actors(df)
print("Grouping applied.")
print(f"Rows: {len(df):,}  |  Columns added: side, coalition, actor_group, is_mapped")


In [ ]:
# ── Validation 1: distribution across the three levels ─────────────────────
print("═" * 60)
print("SIDE")
print("═" * 60)
print(df["side"].value_counts().to_string())

print("\n" + "═" * 60)
print("COALITION")
print("═" * 60)
print(df["coalition"].value_counts().to_string())

print("\n" + "═" * 60)
print("ACTOR_GROUP")
print("═" * 60)
print(df["actor_group"].value_counts().to_string())

print("\n" + "═" * 60)
print("IS_MAPPED")
print("═" * 60)
print(df["is_mapped"].value_counts().to_string())
print(f"\nMapped rate: {df['is_mapped'].mean():.1%}")


In [ ]:
# ── Validation 2: hierarchy consistency check ───────────────────────────────
print("Crosstab side × coalition (event counts):")
xtab = pd.crosstab(df["side"], df["coalition"])
print(xtab.to_string())

# Spot-check: every coalition belongs to exactly one side
hier = df.groupby("coalition")["side"].nunique()
bad = hier[hier > 1]
if bad.empty:
    print("\n✓ All coalitions map to exactly one side")
else:
    print(f"\n✗ Coalitions with multiple sides (hierarchy break): {bad.to_dict()}")


In [ ]:
# ── Validation 3: top-5 raw actor1 strings per actor_group ─────────────────
# This is the proof that regex matching is doing the right thing.
print("Top-5 raw actor1 strings per actor_group\n")
for ag in df["actor_group"].value_counts().index:
    sub = df[df["actor_group"] == ag]["actor1"].value_counts().head(5)
    n_total = (df["actor_group"] == ag).sum()
    print(f"  [{ag}]  ({n_total:,} events)")
    for actor, cnt in sub.items():
        print(f"    {cnt:5d}  {actor}")
    print()


In [ ]:
# ── Validation 4: high-frequency unmapped actors (> 50 events) ──────────────
unmapped_df = df[df["actor_group"] == "Unmapped"]
print(f"Total unmapped events: {len(unmapped_df):,} ({len(unmapped_df)/len(df):.1%})")
print()
high_freq = unmapped_df["actor1"].value_counts()
high_freq = high_freq[high_freq > 50]
if high_freq.empty:
    print("No unmapped actors with > 50 events.")
else:
    print("Unmapped actors with > 50 events (candidates for manual rule additions):")
    for actor, cnt in high_freq.items():
        inter1_val = df[df["actor1"] == actor]["inter1"].mode()[0]
        print(f"  {cnt:5d}  [{inter1_val}]  {actor}")


In [ ]:
# ── Validation 5: recent-period actor_group checklist (2025-01-01 onward) ────
import difflib

recent = df[df["event_date"] >= "2025-01-01"]
THRESHOLD = 10

EXPECTED = [
    ("Arakan Army (AA)",                         "Arakan Army"),
    ("Chin Brotherhood Alliance",                "Chin Brotherhood Alliance"),
    ("Chinland Council",                         "Chinland Council"),
    ("ZRA",                                      "ZRA"),
    ("People's Defence Force",                   "People's Defence Force"),
    ("Tatmadaw",                                 "Tatmadaw"),
    ("PNA",                                      "PNA"),
    ("PNLA",                                     "PNLA — date-gated; pre-2024 to Unmapped; 0 events 2025+"),
    ("Karen National Liberation Army (KNLA)",    "Karen National Liberation Army"),
    ("Mon National Liberation Army (MNLA)",      "Mon National Liberation Army"),
    ("RCSS / Shan State Army South",             "RCSS / Shan State Army South"),
    ("TNLA",                                     "TNLA"),
    ("UWSA",                                     "UWSA"),
    ("NDAA",                                     "NDAA"),
    ("Kachin Independence Army (KIA)",           "Kachin Independence Army"),
    ("Karenni Army",                             "Karenni Army"),
    ("KNDF",                                     "KNDF"),
]

print("Expected actor_groups in recent period (2025+):\n")
all_recent_groups = recent["actor_group"].value_counts()
for ag, label in EXPECTED:
    n = all_recent_groups.get(ag, 0)
    found = n >= THRESHOLD
    mark = "✓" if found else "✗"
    note = f"(n={n})" if n > 0 else "(NOT FOUND)"
    # Append design notes
    if ag == "PNLA" and n < THRESHOLD:
        note += " — by design: date-gated to 2024-01-01; only 15 events in 2024, 0 in 2025+"
    elif ag == "NDAA" and n < THRESHOLD:
        note += " — only 5 events total in dataset (PSC/NDAA-ESS very sparse)"
    elif ag == "ZRA" and n < THRESHOLD:
        note += f" — {all_recent_groups.get(ag, 0)} events; pro-junta EAO, small presence"
    if not found and ag not in ["PNLA", "NDAA", "ZRA"]:
        # Suggest closest actor1 strings via difflib
        candidates = difflib.get_close_matches(
            label, recent["actor1"].dropna().unique().tolist(), n=3, cutoff=0.4
        )
        if candidates:
            note += f" — closest actor1: {candidates}"
    print(f"  {mark} {label:<40s} {note}")

# Karen IEC — by design folded into KNLA
print(f"  ✗ Karen IEC                              (folded into KNLA by design — no separate entry needed)")


### Actor Grouping — Single Source of Truth

This is the canonical actor grouping for all downstream Myanmar analysis. Do not redefine these mappings in other notebooks — import from here or copy `ACTOR_RULES`, `TIME_RULES`, and `INTER1_FALLBACK`.

#### Hierarchy

```
side (4 values)
├── Anti-junta
│   ├── National Unity Government (PDF, NUG)
│   ├── Northern Alliance
│   ├── Three Brotherhood Alliance
│   ├── 4K Coalition
│   ├── Chinland Council
│   ├── Chin Brotherhood Alliance
│   ├── Spring Revolution Alliance
│   ├── Other allied EAOs
│   └── Other anti-junta organisations
├── Pro-junta
│   ├── Tatmadaw and allies
│   └── Aligned ethnic armed organisations
├── Neutral / Self-administered
│   └── Self-administered zones
└── Other / Unmapped
    ├── Other combatant  (time-expired alliance actors)
    └── Unmapped
```

#### Final value counts (run cell above to get live numbers)

- `side`: 4 values
- `coalition`: ~12 values
- `actor_group`: 30+ values (higher than initial ~20-25 estimate; Myanmar's fragmentation requires granularity)

#### Time-varying rules

| actor_group | Cutoff date | Before cutoff | After cutoff |
|---|---|---|---|
| TNLA | 2026-01-01 | Anti-junta / Three Brotherhood Alliance | Other / Unmapped / Other combatant |
| MNDAA | 2026-01-01 | Anti-junta / Three Brotherhood Alliance | Other / Unmapped / Other combatant |
| KNPLF | 2023-01-01 | Other / Unmapped / Unmapped | Anti-junta / 4K Coalition |
| PNLA | 2024-01-01 | Other / Unmapped / Unmapped | Anti-junta / Spring Revolution Alliance |

#### Fallback behavior

If `actor1` matches no named rule, `inter1` type is used: State forces -> Tatmadaw; Civilians -> Civilians; Protesters/Rioters -> Protesters; Rebels/Militias/Identity militias to Unmapped (Other combatant). `is_mapped = False` for all fallback rows.

#### Unmapped actors to watch

Any actor with > 50 events and `is_mapped = False` is printed in Validation 4 above and should be reviewed for manual rule additions in future iterations.

In [ ]:
from pathlib import Path

CKPT_DIR = Path("../checkpoints")
CKPT_DIR.mkdir(exist_ok=True)
CKPT_PATH = CKPT_DIR / "myanmar_acled_grouped.parquet"

df.to_parquet(CKPT_PATH, index=False)
print(f"Checkpoint saved: {CKPT_PATH.resolve()}")
print(f"Rows: {len(df):,}  |  Columns: {list(df.columns)}")
